# Compare all KT runs

Este notebook lê os resultados padronizados salvos em `artifacts/colab_summaries/`
e compara as métricas dos 9 experimentos.

## Objetivo
Permitir uma visão consolidada de:
- AUC de validação e teste
- Loss de validação e teste
- Accuracy de teste
- diferenças entre modelos e tarefas


## Etapa 1 — Setup e localização do diretório de summaries


In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd
import matplotlib.pyplot as plt

IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/GuilhermeDesoler/ai-core.git'

if IN_COLAB:
    REPO_DIR = Path('/content/ai-core')
    if not REPO_DIR.exists():
        get_ipython().system(f'git clone -b improve/high-impact-training {REPO_URL} /content/ai-core')
    get_ipython().run_line_magic('cd', '/content/ai-core')
    get_ipython().system('pip install -q -r requirements.txt')
else:
    cwd = Path.cwd().resolve()
    if cwd.name == 'colab' and cwd.parent.name == 'notebooks':
        REPO_DIR = cwd.parents[1]
    elif cwd.name == 'notebooks':
        REPO_DIR = cwd.parent
    else:
        REPO_DIR = cwd

summary_dir = REPO_DIR / 'artifacts' / 'colab_summaries'
print('Summary dir:', summary_dir)
print('Exists:', summary_dir.exists())


## Etapa 2 — Carregar métricas consolidadas


In [ ]:
rows = []
for p in sorted(summary_dir.glob('*_metrics.json')):
    data = json.loads(p.read_text()) if p.exists() else {}
    name = p.stem.replace('_metrics', '')
    model, task = name.split('_', 1)
    rows.append(
        {
            'name': name,
            'model': model.upper(),
            'task': task,
            'best_val_auc': data.get('best_val_auc'),
            'best_val_loss': data.get('best_val_loss'),
            'test_auc': data.get('test_auc'),
            'test_loss': data.get('test_loss'),
            'test_acc': data.get('test_acc'),
        }
    )

results_df = pd.DataFrame(rows).sort_values(['task', 'model'])
results_df


## Etapa 3 — Comparar métricas com tabelas e gráficos


In [ ]:
for metric in ['test_auc', 'test_loss', 'test_acc']:
    pivot = results_df.pivot(index='task', columns='model', values=metric)
    display(pivot)
    fig = plt.figure(figsize=(10, 5))
    pivot.plot(kind='bar', ax=plt.gca())
    plt.title(f'Comparison - {metric}')
    plt.ylabel(metric)
    plt.grid(alpha=0.3)
    plt.show()
